# Chapter 2 — Computing with vectors and matrices · Solution

*Maths for AI, Step by Step*, Belk Siri — companion notebook.

[Open in Google Colab](https://colab.research.google.com/github/belksiri/books/blob/main/maths-pour-l-ia-pas-a-pas/notebooks/en/ch02_corrige.ipynb) · Companion page: https://belksiri.github.io/books/maths-pour-l-ia-pas-a-pas/en.html

Each answer starts with the reasoning on paper, then shows its equivalent in Python.

**Prediction rule of the chapter** (given, not fitted): weights $\mathbf{w} = (3, -8)$ in k€/m² and k€/km, intercept $b = 45$ k€.

With the districts (lesson 2.4), the rule becomes $\mathbf{w} = (3, -8, 12, -10)$, $b = 45$ k€, with Gare (the station) as the reference district.

**Contents.** Data · The lesson computations · Exercises 2.1 to 2.6

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Colors of the book: one color = one role (observed, predicted, deviation, colleague's line)
BLUE, ORANGE, RESIDUAL, GREEN = '#2a78d6', '#eb6834', '#c2185b', '#2e8b57'

## Data

The identifiers (D2.1, D2.E2) are those of the book and of the CSV files.

In [ ]:
# Chapter 2 data, copied from the book's CSV files (do not modify)

# D2.1 — the seven homes with their distance to the center (lessons 2.1 to 2.3)
homes = pd.DataFrame({
    'home':    ['A', 'B', 'C', 'D', 'E', 'F', 'G'],
    'floor_area':  [32, 45, 54, 70, 85, 98, 62],
    'distance': [3.0, 1.5, 2.5, 1.0, 2.5, 0.5, 2.5],
    'price':   [118, 172, 185, 245, 279, 335, 209],
})

# D2.4 — five homes in the east of the city, with their district (lesson 2.4)
east = pd.DataFrame({
    'home':    ['K', 'L', 'M', 'N', 'O'],
    'district':    ['Parc', 'Rocade', 'Gare', 'Parc', 'Rocade'],
    'floor_area':  [60, 60, 60, 45, 80],
    'distance': [2.0, 2.0, 2.0, 1.0, 3.0],
    'price':   [223, 200, 210, 186, 248],
})

# D2.5 — the five two-room apartments (T2) in the South with their distance to the center (lesson 2.5)
t2_south = pd.DataFrame({
    'sale':       ['S03', 'S04', 'S05', 'S06', 'S07'],
    'floor_area':  [43, 40, 44, 46, 52],
    'distance': [3.5, 2.0, 1.5, 2.5, 0.5],
})

# D2.E2 — three homes put up for sale (exercise 2.2)
new_homes = pd.DataFrame({
    'home':    ['H', 'I', 'J'],
    'floor_area':  [40, 75, 90],
    'distance': [0.5, 3.5, 1.0],
})

### Data variables

Code names are in English. For the columns, the last column gives the name used in the book's CSV files.

| Name | Meaning | Unit | CSV column |
|:--|:--|:--|:--|
| `home` | home identifier (A, B…) | — | `logement` |
| `floor_area` | living area of the home | m² | `surface_m2` |
| `price` | sale price | k€ | `prix_kEUR` |
| `distance` | distance to the city center | km | `distance_km` |
| `district` | district of the home | — | `quartier` |
| `sale` | sale identifier (S01…) | — | `vente` |

## The lesson computations, in Python

These cells redo the computations of lessons 2.1 to 2.5. Run them and compare with the book.

In [ ]:
# Lesson 2.1 — the agency's rule applied to G, term by term (equation 2.1)
area_part = 3 * 62        # k€/m² × m² = k€
distance_part = -8 * 2.5     # k€/km × km = k€
fixed_part = 45               # k€
price_G = area_part + distance_part + fixed_part
print(area_part, distance_part, fixed_part, '→', price_G, 'k€; signed deviation:', 209 - price_G, 'k€')

In [ ]:
# Lesson 2.2 — vectors, dot product (equation 2.2) and prediction (equation 2.3)
x_G = np.array([62, 2.5])    # (floor area in m², distance in km)
w = np.array([3, -8])        # (k€/m², k€/km), same order as x
b = 45                       # k€
print('dimension of x_G:', x_G.shape, '  w · x_G =', np.dot(w, x_G), '  ŷ =', np.dot(w, x_G) + b)
print('weights in the wrong order:', np.dot(w[::-1], x_G) + b, 'k€ (absurd)')

In [ ]:
# Lesson 2.3 — all predictions at once (equation 2.5)
X = homes[['floor_area', 'distance']].to_numpy(dtype=float)
y = homes['price'].to_numpy(dtype=float)
y_hat = X @ w + b
print('X :', X.shape, ' w :', w.shape, ' ŷ :', y_hat.shape)
print('ŷ =', y_hat)
print('signed deviations y − ŷ =', y - y_hat)
print('mean absolute deviation, 7 homes:', round(np.mean(np.abs(y - y_hat)), 2), 'k€')
print('same, A to F:', round(np.mean(np.abs(y - y_hat)[:6]), 2), 'k€; line from the chapter opening, A to F:',
      round(np.mean(np.abs(y - (3 * X[:, 0] + 30))[:6]), 2), 'k€')

In [ ]:
# Lesson 2.4 — indicator columns (equation 2.6), with Gare (the station) as the reference
dummies = pd.get_dummies(east['district'], drop_first=True).astype(int)   # columns Parc, Rocade
X_east = np.column_stack([east['floor_area'], east['distance'], dummies['Parc'], dummies['Rocade']])
w_east = np.array([3, -8, 12, -10])
y_east = east['price'].to_numpy(dtype=float)
print(dummies.columns.tolist(), ' X:', X_east.shape)
print('with the district:', X_east @ w_east + 45)
print('without the district:', X_east[:, :2] @ w_east[:2] + 45)
print('mean absolute deviation: without', np.mean(np.abs(y_east - (X_east[:, :2] @ w_east[:2] + 45))),
      ' with', np.mean(np.abs(y_east - (X_east @ w_east + 45))))

In [ ]:
# Lesson 2.5 — the same prediction on two scales (equation 2.7), division by n
X5 = t2_south[['floor_area', 'distance']].to_numpy(dtype=float)
mu, sigma = X5.mean(axis=0), X5.std(axis=0, ddof=0)
w, b = np.array([3, -8]), 45
w_std, b_std = w * sigma, b + np.sum(w * mu)
Z = (X5 - mu) / sigma
print('means:', mu, ' standard deviations:', sigma)
print('standardized weights:', w_std, ' intercept:', b_std)
print('original units:', X5 @ w + b)
print('standardized scale:', Z @ w_std + b_std)

## Exercise 2.1 — Reading a diagram

*Concepts: product of a matrix and a vector (lesson 2.3).* Go back to the matrix diagram of lesson 2.3 (the seven homes A to G). The cell below rebuilds $X$, $\mathbf{w}$ and $b$ to check your answers.

In [ ]:
X = homes[['floor_area', 'distance']].to_numpy(dtype=float)
w, b = np.array([3, -8]), 45
print(X)

In [ ]:
dimensions = {'X': (7, 2), 'w': (2,), 'y_hat': (7,)}
row_280 = 'E'
dimensions_c = {'X': (8, 3), 'w': (3,), 'y_hat': (8,)}

In [ ]:
# Check
y_hat = X @ w + b
truth = {'X': X.shape, 'w': w.shape, 'y_hat': y_hat.shape}
if None in dimensions.values() or row_280 is None or None in dimensions_c.values():
    print('Complete the previous cell.')
else:
    print('a) dimensions:', 'correct' if dimensions == truth else 'check again')
    letter = homes['home'][int(np.argmin(np.abs(y_hat - 280)))]
    print('b) home:', 'correct' if row_280 == letter else 'check again')
    X8 = np.zeros((8, 3)); w3 = np.zeros(3)   # shape only: 8 homes, 3 features
    print('c) dimensions:', 'correct' if dimensions_c == {'X': X8.shape, 'w': w3.shape,
          'y_hat': (X8 @ w3).shape} else 'check again')

### Solution to exercise 2.1

**a)** $X$: **7 × 2** (7 homes, 2 features). $\mathbf{w}$: **2** components. $\hat{\mathbf{y}}$: **7** predictions.

**b)** It is row **E**: 85 m² and 2.5 km. With $\mathbf{w} = (3, -8)$ and $b = 45$: 3 × 85 + (−8) × 2.5 + 45 = 255 − 20 + 45 = **280 k€**.

**c)** With the floor level and an eighth home: $X$ becomes **8 × 3**, $\mathbf{w}$ has **3** components (one weight for the floor level), $\hat{\mathbf{y}}$ has **8** predictions. If you only add a home, $\mathbf{w}$ keeps its **2** components: the number of weights depends on the features, not on the number of homes. $X$ then becomes 8 × 2 and $\hat{\mathbf{y}}$ has 8 predictions.

*Expected elements:* the three dimensions; the five numbers used for E; the distinction between rows (homes) and columns (features).

## Exercise 2.2 — Three new homes

*Data D2.E2. Concepts: dot product (lesson 2.2), product of a matrix and a vector (lesson 2.3).* Do questions a) to c) on paper first, with the table of exercise 2.2 in the book.

In [ ]:
new_homes

### a) Write

$\mathbf{x}_H = (40, 0.5)$, $\mathbf{x}_I = (75, 3.5)$, $\mathbf{x}_J = (90, 1.0)$, with the floor area in m² first, then the distance in km. The matrix stacks these vectors as rows, in the same order:

$$X = \begin{pmatrix} 40 & 0.5 \\ 75 & 3.5 \\ 90 & 1.0 \end{pmatrix}$$

It has 3 rows (one per home) and 2 columns (one per feature): its dimensions are **3 × 2**.

### b) Compute by hand

| Home | floor area: 3 k€/m² × … | distance: −8 k€/km × … | fixed part | predicted price |
|:--:|:--|:--|--:|--:|
| H | 3 × 40 = 120 k€ | −8 × 0.5 = −4 k€ | 45 k€ | **161 k€** |
| I | 3 × 75 = 225 k€ | −8 × 3.5 = −28 k€ | 45 k€ | **242 k€** |
| J | 3 × 90 = 270 k€ | −8 × 1.0 = −8 k€ | 45 k€ | **307 k€** |

Each term is in k€: the m² cancels with the "per m²", the km with the "per km".

In [ ]:
price_by_hand = {'H': 161, 'I': 242, 'J': 307}

In [ ]:
# Check (independent computation, row by row)
for _, l in new_homes.iterrows():
    truth = 3 * l['floor_area'] - 8 * l['distance'] + 45
    yours = price_by_hand[l['home']]
    status = 'not yet completed' if yours is None else ('correct' if np.isclose(yours, truth) else 'check again')
    print(f"{l['home']} : {status}")

### c) Interpret

Line from the chapter opening (3 × floor area + 30): H → 150 k€, I → 255 k€, J → 300 k€. Differences (new rule − line): H **+11**, I **−13**, J **+7** k€. The prediction changes most for **I**, the home farthest from the center (3.5 km). It drops by 13 k€.

*Going further.* The difference between the two rules is $(3x_1 - 8x_2 + 45) - (3x_1 + 30) = 15 - 8x_2$: it depends only on the distance. It is positive below 15/8 ≈ 1.9 km and negative beyond. H (0.5 km) gains 15 − 4 = 11 k€, I (3.5 km) loses 28 − 15 = 13 k€, J (1 km) gains 15 − 8 = 7 k€.

*Expected elements:* the three prices from the line; the three differences; I identified and linked to its distance.

### d) All predictions in one line

Complete the cell, then run the check.

In [ ]:
X = new_homes[['floor_area', 'distance']].to_numpy(dtype=float)   # 3 × 2
w = np.array([3, -8])    # k€/m², k€/km
b = 45                   # k€
y_hat = X @ w + b    # equation 2.5

`X @ w` computes the three dot products (one row of $X$ with $\mathbf{w}$ each time). NumPy then adds $b$ to each of them.

In [ ]:
# Check
if y_hat is None:
    print('Complete the previous cell.')
else:
    print('dimensions:', np.shape(X), '@', np.shape(w), '→', np.shape(y_hat))
    print('dimensions:', 'correct' if np.shape(X) == (3, 2) and np.shape(y_hat) == (3,) else 'check again')
    print('predictions:', y_hat, '→',
          'correct' if np.allclose(y_hat, [161, 242, 307]) else 'check again')

### e) Predict, then check

**Prediction.** The weight −8 now multiplies the floor area (tens of m²) and the weight 3 the distance (a few km). The term −8 × floor area wins by far: the three prices will be **negative**.

**On paper.** H: 3 × 0.5 − 8 × 40 + 45 = 1.5 − 320 + 45 = −273.5 k€. I: 3 × 3.5 − 8 × 75 + 45 = 10.5 − 600 + 45 = −544.5 k€. J: 3 × 1 − 8 × 90 + 45 = 3 − 720 + 45 = −672 k€.

**Why no error?** NumPy only checks the dimensions: 3 × 2 multiplied by 2 gives 3. They are always compatible, and NumPy knows neither the meaning nor the unit of the columns. It is up to you to keep the same order in $X$ and in $\mathbf{w}$. A simple check is to verify that a predicted price is plausible, or to keep the column names with pandas.

In [ ]:
if y_hat is not None:
    X_inverse = X[:, ::-1]          # columns swapped: distance, floor area
    print('columns swapped:', X_inverse @ w + b)

## Exercise 2.3 — What do the weights say?

*Concepts: reading a weight (lesson 2.2), indicator columns (lesson 2.4).* Rule: $\mathbf{w} = (3, -8, 12, -10)$ for floor area, distance, Parc, Rocade; $b = 45$ k€; Gare is the reference. Questions on paper, check below.

In [ ]:
a_park_minus_ring_road = 12 - (-10)        # = 22 k€
b_far_minus_near = -8 * 1.5     # = −12 k€
c_far_park_minus_station = 12 - 8 * 1.5   # = 0 k€

In [ ]:
# Check: we compute the rule for pairs of fictitious homes
rule = lambda floor_area, distance, park, ring_road: 3 * floor_area - 8 * distance + 12 * park - 10 * ring_road + 45
truths = {'a': rule(60, 2, 1, 0) - rule(60, 2, 0, 1),
         'b': rule(60, 3.5, 0, 0) - rule(60, 2, 0, 0),
         'c': rule(60, 3.5, 1, 0) - rule(60, 2, 0, 0)}
for k, yours in [('a', a_park_minus_ring_road), ('b', b_far_minus_near), ('c', c_far_park_minus_station)]:
    status = 'not yet completed' if yours is None else ('correct' if np.isclose(yours, truths[k]) else 'check again')
    print(f'{k}) {status}')

### Solution to exercise 2.3

**a)** Only the district terms differ: $12 \times 1 - (-10) \times 1 = 12 + 10 =$ **22 k€** more in Parc than in Rocade.

**b)** Only the distance term differs: $-8 \text{ k€/km} \times 1.5 \text{ km} =$ **−12 k€**. The farther apartment is predicted 12 k€ cheaper.

**c)** The two effects cancel out: +12 k€ for Parc, −12 k€ for the extra 1.5 km. The rule predicts **the same price**. Each weight is read "all other things being equal". When two features change at once, you add their effects.

**d)** $b = 45$ k€ is the predicted price for a home of 0 m², located 0 km from the center, in the reference district (Gare). Such a home does not exist: $b$ sets the level of the rule, like the 30 k€ of the chapter opening.

*Expected elements:* the three differences with their unit; the idea of compensation; reading the district weights "relative to Gare"; $b$ with no physical reality.

## Exercise 2.4 — Indicator columns in Python

*Data D2.4. Concepts: indicator columns (lesson 2.4), product of a matrix and a vector (lesson 2.3).*

In [ ]:
east

In [ ]:
park = (east['district'] == 'Parc').astype(int).to_numpy()
ring_road = (east['district'] == 'Rocade').astype(int).to_numpy()
X4 = np.column_stack([east['floor_area'], east['distance'], park, ring_road])
w4 = np.array([3, -8, 12, -10])
b4 = 45
y4 = X4 @ w4 + b4

**Variant.** `pd.get_dummies(east['district'], drop_first=True)` directly creates the Parc and Rocade columns, dropping Gare, the first category in alphabetical order. Always check which category was dropped: it is the one that serves as the reference for reading the weights.

In [ ]:
# Check
if y4 is None:
    print('Complete the previous cell.')
else:
    print('indicator columns:', 'correct' if list(park) == [1, 0, 0, 1, 0] and list(ring_road) == [0, 1, 0, 0, 1] else 'check again')
    print('X4 :', np.shape(X4), '→', 'correct' if np.shape(X4) == (5, 4) else 'check again')
    print('predictions:', y4, '→', 'correct' if np.allclose(y4, [221, 199, 209, 184, 251]) else 'check again')

**c)** Mean absolute deviation, with and without the district columns.

In [ ]:
y_obs = east['price'].to_numpy(dtype=float)
mae_without = np.mean(np.abs(y_obs - (X4[:, :2] @ w4[:2] + b4)))
mae_with = np.mean(np.abs(y_obs - y4))

In [ ]:
if mae_without is None or mae_with is None:
    print('Complete the previous cell.')
else:
    print(f'without: {mae_without:.1f} k€, with: {mae_with:.1f} k€ →',
          'correct' if np.isclose(mae_without, 10.2) and np.isclose(mae_with, 1.8) else 'check again')

### Solution to exercise 2.4

**a)** Parc: (1, 0, 0, 1, 0). Rocade: (0, 1, 0, 0, 1). M, in Gare, has two zeros.

**b)** $\hat{\mathbf{y}} = (221, 199, 209, 184, 251)$ k€. For example for N: 3 × 45 − 8 × 1 + 12 × 1 − 10 × 0 + 45 = 135 − 8 + 12 + 45 = 184 k€.

**c)** Without the district, the signed deviations are +14, −9, +1, +14, −13: mean of the absolute values 51 / 5 = **10.2 k€**. With the district: +2, +1, +1, +2, −3, i.e. 9 / 5 = **1.8 k€**. As in the lesson, these numbers describe these five sales. They do not measure what the rule would give on future sales.

## Exercise 2.5 — Predict, then check: changing units

*Data D2.5. Concepts: change of scale (lesson 2.5, equation 2.7).* Write down your predictions for a) and b) before running the cells.

**Expected predictions.** a) The weight of the distance becomes −0.008 k€ per meter. The intercept stays at 45 k€, and the five predictions do not change. b) The mean goes from 2 km to 2,000 m, the standard deviation from 1 km to 1,000 m. The standardized weight stays $w'_2 = -0.008 \times 1{,}000 = -8$ k€ per standard deviation: it does not depend on the chosen unit.

In [ ]:
def standardized_weights(w, b, mu, sigma):
    w_std = w * sigma              # k€ per standard deviation
    b_std = b + np.sum(w * mu)     # predicted price of the average home
    return w_std, b_std

In [ ]:
# Two versions of the same rule: distance in km, then in meters
X_km = t2_south[['floor_area', 'distance']].to_numpy(dtype=float)
X_m = X_km * np.array([1, 1000])
w_km, w_m, b = np.array([3, -8]), np.array([3, -0.008]), 45
print('predictions (km):', X_km @ w_km + b)
print('predictions (m): ', X_m @ w_m + b)
results = {}
for name, Xv, wv in [('km', X_km, w_km), ('m', X_m, w_m)]:
    mu, sigma = Xv.mean(axis=0), Xv.std(axis=0, ddof=0)
    results[name] = standardized_weights(wv, b, mu, sigma)
    print(name, ': mean distance =', mu[1], ', standard deviation =', sigma[1], ', standardized weights =', results[name])

In [ ]:
# Check
if results['km'][0] is None:
    print('Complete the function standardized_weights.')
else:
    ok = all(np.allclose(results[k][0], [12, -8]) and np.isclose(results[k][1], 164) for k in results)
    print('standardized weights identical in km and in m:', 'correct' if ok else 'check again')

In [ ]:
if results['km'][0] is not None:
    fig, axes = plt.subplots(1, 2, figsize=(8, 2.6))
    for ax, title, values in [
            (axes[0], 'Raw weight of the 2nd term', {'km': w_km[1], 'm': w_m[1]}),
            (axes[1], 'Standardized weight of the 2nd term', {k: results[k][0][1] for k in results})]:
        ax.bar(list(values), list(values.values()), color=[BLUE, ORANGE])
        ax.axhline(0, color='black', linewidth=0.8)
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()

### Solution to exercise 2.5

**a)** One meter is 1/1,000 km: the weight becomes −8 / 1,000 = **−0.008 k€/m**. Since each distance is multiplied by 1,000, each product $w_2 x_2$ stays the same: the intercept (45 k€) and the five predictions (146, 149, 165, 163, 197 k€) **do not change**.

**b)** Mean 2,000 m, standard deviation 1,000 m. With equation 2.7: $w'_2 = -0.008 \times 1{,}000 = -8$ and $b' = 45 + 3 \times 45 - 0.008 \times 2{,}000 = 45 + 135 - 16 = 164$ k€, **exactly** as in km.

**c)** On the left-hand chart, the raw weight goes from −8 to −0.008: its size depends on the unit and allows no comparison. On the right-hand one, the standardized weights coincide: an increase of one standard deviation in the distance always corresponds to −8 k€ in the prediction, with the floor area held fixed. This reading describes the rule on these data. It measures neither a general importance nor a causal effect.

*Expected elements:* predictions unchanged; raw weight divided by 1,000; standardized weight unchanged; conclusion about comparing weights.

## Exercise 2.6 — Challenge: one rule, several ways to write it

*Data D2.4. Concepts: lessons 2.3, 2.4 and 2.5.* Three indicator columns: Parc, Rocade, Gare.

In [ ]:
station = (east['district'] == 'Gare').astype(int).to_numpy()
park3 = (east['district'] == 'Parc').astype(int).to_numpy()
ring_road3 = (east['district'] == 'Rocade').astype(int).to_numpy()
X3 = np.column_stack([east['floor_area'], east['distance'], park3, ring_road3, station])
print('sum of the three indicators, row by row:', X3[:, 2:].sum(axis=1))
print('the colleague’s rule (b = 40):', X3 @ np.array([3, -8, 17, -5, 5]) + 40)

In [ ]:
district_weight_b50 = [7, -15, -5]   # each district weight decreases by 5 when b increases by 5

In [ ]:
# Check
if district_weight_b50 is None:
    print('Complete the previous cell.')
else:
    y50 = X3 @ np.array([3, -8] + list(district_weight_b50)) + 50
    print('predictions with b = 50:', y50, '→', 'correct' if np.allclose(y50, [221, 199, 209, 184, 251]) else 'check again')

### Solution to exercise 2.6

**a)** K: 180 − 16 + 17 + 40 = 221; L: 180 − 16 − 5 + 40 = 199; M: 180 − 16 + 5 + 40 = 209; N: 135 − 8 + 17 + 40 = 184; O: 240 − 24 − 5 + 40 = 251. These are **exactly** the predictions in the table of lesson 2.4.

**b)** Each home has one and only one 1 among the three indicators. Adding the same amount $c$ to the three district weights therefore adds $c$ to all predictions, which you compensate by subtracting $c$ from $b$. For $b = 50$, $c = -5$ relative to the rule of the lesson (12, −10, 0, $b = 45$): weights **(7, −15, −5)**. Any $c$ works: there are infinitely many equivalent rules.

**c)** In equation 2.7, you change the *columns* (original units or standardized scores), and each way of writing the columns corresponds to a single rule. It is a translation, with no ambiguity. With three indicators, the columns stay the same and yet infinitely many weights give the same predictions. It is this second situation that hinders the fitting in chapter 3: nothing in the data lets you choose between these rules. Removing one indicator, the one for the reference, removes the ambiguity.

*Expected elements:* the five predictions; the argument that the indicators sum to 1; the distinction between change of scale (unique rule for given columns) and redundancy (non-unique rule).